# Models evaluation

thesis prooject model evalutaion

In [ ]:
LOCAL_RUN = True  

PATH_IMG = "images"

if LOCAL_RUN:
    PATH = "DATASETS"
    PERCATANGE_TO_USE = 0.1
else:
    PATH = "file:///home/PuckTrickadmin/DATASETS"
    PERCATANGE_TO_USE = 1.0

## 0. setup

In [3]:
!pip install -r requirements.txt
%pip install setuptools yellowbrick


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# import delle librerie necessarie
import numpy as np
import pandas as pd  # used for toPandas() visualization conversions
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans Nabataean, Nimbus Sans, Nimbus Roman'
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, when, isnan
# split dataset in train and test set
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
# RFECV - no direct Spark equivalent; keeping sklearn for cross-validation strategies
from sklearn.model_selection import StratifiedKFold
import os
import glob
from sklearn.metrics import matthews_corrcoef, make_scorer
import seaborn as sns
import os
import subprocess
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.sql.functions import to_timestamp, date_format, hour, dayofweek, window, year
from pyspark.sql.functions import abs as spark_abs
from pyspark.sql.functions import col as _col

In [5]:
if(LOCAL_RUN):
    # Check Java installation
    java_home = os.environ.get('JAVA_HOME', '')
    if not java_home:
        # Try to find Java automatically
        try:
            java_path = subprocess.check_output(['which', 'java'], text=True).strip()
            java_home = os.path.dirname(os.path.dirname(os.path.realpath(java_path)))
            os.environ['JAVA_HOME'] = java_home
            print(f"Found Java at: {java_home}")
        except subprocess.CalledProcessError:
            print("Java not found! Please install Java 8 or 11.")
            print("Run: sudo apt install default-jdk")

    os.environ['PYSPARK_PYTHON'] = 'python3'
    os.environ['PYSPARK_DRIVER_PYTHON'] = 'python3'

    # Initialize Spark session
    spark = SparkSession.builder \
        .appName("PucktrickDataQuality") \
        .master("local[*]") \
        .config("spark.driver.memory", "14g") \
        .config("spark.driver.host", "localhost") \
        .config("spark.ui.showConsoleProgress", "false") \
        .getOrCreate()

    print(f"Spark version: {spark.version}")

Found Java at: /usr/lib/jvm/java-11-openjdk


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 17:56:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.0


In [6]:
if not LOCAL_RUN:
    MASTER_URL   = "spark://10.0.1.8:7077"
    DRIVER_HOST  = "10.0.1.8"

    spark = SparkSession.builder \
        .appName("Cavas_Analysis") \
        .master(MASTER_URL) \
        .config("spark.submit.deployMode",      "client") \
        .config("spark.executor.instances",     "4") \
        .config("spark.executor.cores",         "4") \
        .config("spark.executor.memory",        "13g") \
        .config("spark.driver.memory",          "8g") \
        .config("spark.driver.host",            DRIVER_HOST) \
        .config("spark.driver.bindAddress",     DRIVER_HOST) \
        .config("spark.sql.shuffle.partitions", "32") \
        .getOrCreate()

    spark.sparkContext.setLogLevel("WARN")
    print("SparkSession creata — versione:", spark.version)

## 1. Models hyperparameters optimization

In [7]:
dataset = spark.read.parquet(f'{PATH}/all_elaborated.parquet')

In [8]:
uni_target = dataset.select('Label')
uni_target_generic = dataset.select('Label_generic')

uni_feature = dataset.drop('Label', 'Label_generic')

In [9]:
len(uni_feature.columns)

44